`[1]`

# CNN de favorabilidade de fosfato — treino no Colab

Só o **treino**: as mesmas rodadas do servidor (`rodadas_treino.md`), com
validação cruzada em 5 folds e o teste final. Os scripts são os de
`src/model_DL/`, importados (não copiados); aqui só mudam os caminhos (Drive).

| Etapa | Onde | Células |
|---|---|---|
| base de patches (TFRecord) | máquina local: `python exportar_patches_drive.py` → Drive | — |
| converter TFRecord → `.npz` (uma vez) | aqui: `tfrecord_para_npz.py` | 5 |
| conferir os patches | aqui | 6 |
| treinar uma rodada (5 folds) | aqui: `treinar_cnn.py` | 7 |
| comparar as rodadas | aqui | 8 |
| teste final (uma vez) | aqui: `treinar_cnn.py --teste` | 9 |
| referência × predição em patches de teste | aqui | 10 |
| todas as presenças de fosfato × predição das rodadas (tabela e gráficos) | aqui | 11–12 |
| tiles e mapa | máquina local `exportar_tiles_drive.py` + notebook de predição | — |

**Sem Earth Engine:** a base já está em TFRecord no Drive (`prospectra/patchs_mine`); aqui só
TensorFlow (leitura do TFRecord) e PyTorch/timm (treino). No Drive basta
`prospectra/models_scripts/` com `treinar_cnn.py`, `exportar_patches.py` e
`tfrecord_para_npz.py`.

**Antes:** *Ambiente de execução → Alterar o tipo → GPU* (T4, L4 ou A100).

In [ ]:
# [2] Drive, código do projeto e pastas
from google.colab import drive
drive.mount('/content/drive')

import os, sys, shutil, subprocess
from pathlib import Path

#@markdown Código: pasta com os .py (treinar_cnn, exportar_patches, tfrecord_para_npz) **ou** URL git
SCRIPTS_DIR = '/content/drive/MyDrive/prospectra/models_scripts'  #@param {type:"string"}
GIT_URL = ''  #@param {type:"string"}
#@markdown Base de patches (.npz) e rodadas no Drive
PATCHES_DIR = '/content/drive/MyDrive/prospectra/patches_cnn'  #@param {type:"string"}
RUNS_DIR = '/content/drive/MyDrive/prospectra/runs_cnn'  #@param {type:"string"}
#@markdown Copiar os patches para o disco do Colab (leitura bem mais rápida no treino)
COPIAR_LOCAL = True  #@param {type:"boolean"}

if GIT_URL:
    if not Path('/content/repo').exists():
        subprocess.run(['git', 'clone', '-q', GIT_URL, '/content/repo'], check=True)
    SCRIPTS = Path('/content/repo/src/model_DL')
else:
    SCRIPTS = Path(SCRIPTS_DIR)
# sem Earth Engine: a base já está em TFRecord no Drive; o exportar_patches.py só fornece
# as constantes (canais, modelos, patch) e a gravação dos .npz
for f in ('treinar_cnn.py', 'exportar_patches.py', 'tfrecord_para_npz.py'):
    assert (SCRIPTS / f).exists(), f'falta {SCRIPTS / f} — suba o .py para essa pasta do Drive'

Path(PATCHES_DIR).mkdir(parents=True, exist_ok=True)
Path(RUNS_DIR).mkdir(parents=True, exist_ok=True)
os.environ['PATCHES_DIR'] = PATCHES_DIR
os.environ['RUNS_DIR'] = RUNS_DIR
sys.path.insert(0, str(SCRIPTS))
os.chdir(SCRIPTS)
print('código :', SCRIPTS)
print('patches:', PATCHES_DIR)
print('rodadas:', RUNS_DIR)

In [ ]:
# [3] pacotes (torch, tensorflow e sklearn já vêm no Colab)
%pip install -q timm

In [ ]:
# [4] GPU
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB')
else:
    print('Sem GPU: Ambiente de execução → Alterar o tipo → GPU')

`[4b]`

## Base de patches

A base vem do `exportar_patches_drive.py` (tasks do GEE → TFRecord no Drive).
A célula 5 converte para os `.npz` do treino em `PATCHES_DIR` — uma vez só;
células já convertidas são puladas. Treino, validação e teste ficam juntos: a
divisão está nos campos `conjunto` e `fold` de cada patch (por célula da grid).

In [ ]:
# [5] converter TFRecord (exportar_patches_drive.py) → .npz
#@markdown Pasta com os patches_<LxCy>_<k>.tfrecord.gz, indice_patches.csv e divisao.csv
TFRECORD_DIR = '/content/drive/MyDrive/prospectra/patchs_mine'  #@param {type:"string"}
CONVERTER = True  #@param {type:"boolean"}

import tfrecord_para_npz as tfn
if CONVERTER:
    tfn.main(['--origem', TFRECORD_DIR, '--destino', PATCHES_DIR])
else:
    print('Conversão desmarcada.')

In [ ]:
# [6] conferir os patches: contagens por conjunto e alguns exemplos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import treinar_cnn as tc

MODELO_VER = 'A'  #@param ["A", "B1"]
X, meta, canais = tc.carregar(MODELO_VER, Path(PATCHES_DIR))
print(X.shape, '| canais:', canais)
print(pd.crosstab(meta.origem_amostra, [meta.conjunto, meta.fold]).to_string())

CANAL_VER = canais[0]
c = canais.index(CANAL_VER)
fig, axs = plt.subplots(3, 6, figsize=(15, 8))
for ax, origem in zip(axs, ['ocorrencia_treino', 'ambos', 'negativo']):
    idx = np.where(meta.origem_amostra == origem)[0][:6]
    for a, i in zip(ax, idx):
        a.imshow(X[i, c].astype(np.float32), cmap='magma')
        a.set_title(f'{origem}\ny={meta.y[i]:.2f}', fontsize=8); a.axis('off')
fig.suptitle(f'{MODELO_VER} · canal {CANAL_VER}'); plt.tight_layout()

`[6b]`

## Treino com validação cruzada (5 folds)

As rodadas são as do `rodadas_treino.md` (nome = `<modelo>_<backbone>_<patch>`):
escolha uma e rode; a pasta `RUNS_DIR/<rodada>/` recebe `fold*.pt`,
`historico_fold*.csv`, `metricas_folds.csv`, `oof_validacao.csv` e `config.json`.
Ordem sugerida: as duas `pequena_32` (linha de base), depois os backbones a 64 px.
A métrica que decide é `ap_ponto` (com `ap_ganho`, `boyce`, `top10`, `auc_ponto`;
GTB = 0,84 de `auc_ponto`) — nunca acurácia. Só os folds 0 e 1 têm ocorrências reais.

Refazer uma rodada (divergiu, perda NaN, ou caiu no meio): o script recusa pasta que
já tem `.pt` — apague `RUNS_DIR/<rodada>/` e rode de novo; se divergiu, com
`LR = 3e-4` e `AMP = 0`.

In [ ]:
# [7] treinar uma rodada (validação cruzada)
RODADA = 'A_pequena_32'  #@param ["A_pequena_32", "B1_pequena_32", "A_regnety_64", "B1_regnety_64", "A_effv2s_64", "B1_effv2s_64", "A_resnet50_64", "B1_resnet50_64", "A_coatnet0_64", "B1_coatnet0_64"]
EPOCAS = 40  #@param {type:"integer"}
#@markdown Lote: 64 cabe em 12 GB; na A100 (40 GB) use 256. LR é o de lote 64 —
#@markdown com ESCALAR_LR o lr usado é LR × √(LOTE / 64) (256 → 2×)
LOTE = 256  #@param [64, 128, 256, 512] {type:"raw"}
LR = 1e-3  #@param {type:"number"}
ESCALAR_LR = True  #@param {type:"boolean"}
#@markdown Processos que montam os lotes (CPU); com lote grande a GPU espera por eles
WORKERS = 8  #@param {type:"integer"}
AMP = 1  #@param [1, 0] {type:"raw"}
FOLDS = '0,1,2,3,4'  #@param {type:"string"}

RODADAS = {
    'A_pequena_32': ('A', 'pequena', 32),
    'B1_pequena_32': ('B1', 'pequena', 32),
    'A_regnety_64': ('A', 'regnety_016', 64),
    'B1_regnety_64': ('B1', 'regnety_016', 64),
    'A_effv2s_64': ('A', 'tf_efficientnetv2_s', 64),
    'B1_effv2s_64': ('B1', 'tf_efficientnetv2_s', 64),
    'A_resnet50_64': ('A', 'resnet50', 64),
    'B1_resnet50_64': ('B1', 'resnet50', 64),
    'A_coatnet0_64': ('A', 'coatnet_0_rw_224', 64),
    'B1_coatnet0_64': ('B1', 'coatnet_0_rw_224', 64),
}
import treinar_cnn as tc
MODELO, BACKBONE, PATCH = RODADAS[RODADA]
LR_EF = LR * (LOTE / 64) ** 0.5 if ESCALAR_LR else LR
WORKERS = min(WORKERS, os.cpu_count() or 2)
print(f'{RODADA}: lote {LOTE} · lr {LR_EF:.2e} · {WORKERS} workers')

dados = PATCHES_DIR
if COPIAR_LOCAL:                       # Drive é lento para leitura repetida
    local = Path('/content/patches_cnn')
    if not (local / f'modelo_{MODELO}').exists():
        shutil.copytree(Path(PATCHES_DIR) / f'modelo_{MODELO}', local / f'modelo_{MODELO}')
    dados = str(local)

tc.main(['--modelo', MODELO, '--backbone', BACKBONE, '--patch', str(PATCH), '--nome', RODADA,
         '--epocas', str(EPOCAS), '--lote', str(LOTE), '--lr', str(LR_EF), '--amp', str(AMP),
         '--folds', FOLDS, '--workers', str(WORKERS), '--dados', dados, '--saida', RUNS_DIR])

In [ ]:
# [8] comparar as rodadas: auc_ponto por fold e fora do fold, contra o GTB
from sklearn.metrics import roc_auc_score
linhas = []
import numpy as np
import pandas as pd
import treinar_cnn as tc
for run in sorted(Path(RUNS_DIR).glob('*/')):
    arq = run / 'metricas_folds.csv'
    if not arq.exists():
        continue
    m = pd.read_csv(arq)
    oof = pd.read_csv(run / 'oof_validacao.csv')
    geral = tc.metricas(oof, oof.p.values)
    hist = [pd.read_csv(h) for h in run.glob('historico_fold*.csv')]
    melhor_ep = np.mean([h.loc[h['auc_ponto'].idxmax() if 'auc_ponto' in h and h['auc_ponto'].notna().any()
                               else h['auc_amostra'].idxmax(), 'epoca'] for h in hist]) if hist else np.nan
    linhas.append({'rodada': run.name, 'ap_ponto_oof': geral.get('ap_ponto'),
                   'ap_ganho_oof': geral.get('ap_ganho'), 'boyce_oof': geral.get('boyce'),
                   'top10_oof': geral.get('top10'), 'auc_ponto_oof': geral.get('auc_ponto'),
                   'auc_amostra_oof': geral.get('auc_amostra'), 'brier': geral.get('brier'),
                   'auc_ponto_folds': ', '.join(f'{v:.2f}' for v in m.get('auc_ponto', pd.Series()).dropna()),
                   'melhor_epoca_media': melhor_ep})
TAB = pd.DataFrame(linhas).sort_values('ap_ponto_oof', ascending=False)
TAB.insert(0, 'posicao', range(1, len(TAB) + 1))
# ranking da validação cruzada (fora do fold): o notebook de predição usa a 1ª (RODADA = 'melhor'),
# com o final.pt da <rodada>_teste se ela existir
TAB['tem_teste'] = [(Path(RUNS_DIR) / f'{r}_teste' / 'final.pt').exists() for r in TAB.rodada]
TAB.to_csv(Path(RUNS_DIR) / 'ranking_rodadas.csv', index=False)
print(f'Melhor rodada: {TAB.rodada.iloc[0]}' + ('' if TAB.tem_teste.iloc[0] else
      ' — ainda sem a rodada _teste (célula 9); o mapa usaria a média dos folds'))
print(f'Ordenado por ap_ponto (precisão média, ocorrências × fundo). Referência GTB: auc_ponto = {tc.GTB_REF}')
TAB.round(3)

`[8b]`

## Teste final (uma vez)

Com a melhor rodada da célula 8: treina em todos os folds com número fixo de
épocas (`melhor_epoca_media` da célula 8) e mede no conjunto de teste (células
das ocorrências de validação). Grava em `RUNS_DIR/<rodada>_teste/` (`final.pt`,
`teste.csv`, `metricas_teste.json`). O teste não é olhado durante o treino.

In [ ]:
# [9] teste final
RODADA_T = 'A_regnety_64'  #@param ["A_pequena_32", "B1_pequena_32", "A_regnety_64", "B1_regnety_64", "A_effv2s_64", "B1_effv2s_64", "A_resnet50_64", "B1_resnet50_64", "A_coatnet0_64", "B1_coatnet0_64"]
EPOCAS_T = 20  #@param {type:"integer"}
RODAR_TESTE = False  #@param {type:"boolean"}

import json
import treinar_cnn as tc
MODELO_T, BACKBONE_T, PATCH_T = RODADAS[RODADA_T]       # dicionário da célula 7
# lote, lr e amp: os mesmos da validação cruzada dessa rodada (config.json)
cfg = json.loads((Path(RUNS_DIR) / RODADA_T / 'config.json').read_text())
print(f"{RODADA_T}_teste: lote {cfg['lote']} · lr {cfg['lr']:.2e} · amp {cfg['amp']} (da validação cruzada)")
dados_t = PATCHES_DIR
if COPIAR_LOCAL:
    local = Path('/content/patches_cnn')
    if not (local / f'modelo_{MODELO_T}').exists():
        shutil.copytree(Path(PATCHES_DIR) / f'modelo_{MODELO_T}', local / f'modelo_{MODELO_T}')
    dados_t = str(local)
if RODAR_TESTE:
    tc.main(['--modelo', MODELO_T, '--backbone', BACKBONE_T, '--patch', str(PATCH_T),
             '--epocas', str(EPOCAS_T), '--teste', '--nome', f'{RODADA_T}_teste',
             '--lote', str(cfg['lote']), '--lr', str(cfg['lr']), '--amp', str(cfg['amp']),
             '--workers', str(min(8, os.cpu_count() or 2)),
             '--dados', dados_t, '--saida', RUNS_DIR])
else:
    print('Marque RODAR_TESTE para rodar (uma vez, com a rodada escolhida).')

`[9c]`

## Ver o teste: referência × predição em patches

Pega patches do **conjunto de teste** (células que a rede nunca viu) — metade
ocorrências de fosfato (o ponto SGB, sem jitter) e metade fundo aleatório —, passa
pelo modelo da rodada (`final.pt` da `<rodada>_teste`; sem ela, a média dos
`fold*.pt` da rodada) e mostra cada patch com:

- **referência**: fosfato (ocorrência SGB) ou fundo;
- **p RF / p GTB**: probabilidade dos mapas de referência no centro do patch;
- **p CNN**: probabilidade de fosfato prevista pela rede para o patch.

O ✕ marca o centro (o pixel que recebe o p); o quadrado tracejado, a janela que
a rede vê quando o patch é 32 px.

In [ ]:
# [10] referência × predição em patches de teste
RODADA_V = 'A_regnety_64'  #@param ["A_pequena_32", "B1_pequena_32", "A_regnety_64", "B1_regnety_64", "A_effv2s_64", "B1_effv2s_64", "A_resnet50_64", "B1_resnet50_64", "A_coatnet0_64", "B1_coatnet0_64"]
N_PATCHES = 10  #@param {type:"integer"}
CANAL_V = ''  #@param {type:"string"}
SEMENTE = 0  #@param {type:"integer"}

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader
import treinar_cnn as tc

# modelo: final.pt da rodada de teste; senão, ensemble dos folds da validação cruzada
pasta_t, pasta_cv = Path(RUNS_DIR) / f'{RODADA_V}_teste', Path(RUNS_DIR) / RODADA_V
arquivos = [pasta_t / 'final.pt'] if (pasta_t / 'final.pt').exists() else sorted(pasta_cv.glob('fold*.pt'))
assert arquivos, f'Nenhum final.pt em {pasta_t} nem fold*.pt em {pasta_cv} — treine a rodada antes'
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cks = [torch.load(a, map_location='cpu', weights_only=False) for a in arquivos]
args_m = cks[0]['args']
print(f'modelo: {", ".join(a.name for a in arquivos)} · {args_m["backbone"]} · patch {args_m["patch"]} px')

X_v, meta_v, canais_v = tc.carregar(args_m['modelo'], Path(globals().get('dados', PATCHES_DIR)))
assert canais_v == list(cks[0]['canais']), 'canais dos .npz ≠ canais do modelo'

# patches de teste: metade ocorrências (ponto SGB, jitter 0), metade fundo aleatório
rng = np.random.default_rng(SEMENTE)
te = meta_v[meta_v.conjunto == 'teste']
oc = te[(te.origem_amostra == 'ocorrencia_teste') & (te.jitter == 0)].index.to_numpy()
fu = te[te.origem_amostra == 'fundo'].index.to_numpy()
n_oc = min(len(oc), (N_PATCHES + 1) // 2)
idx = np.concatenate([rng.choice(oc, n_oc, replace=False),
                      rng.choice(fu, min(len(fu), N_PATCHES - n_oc), replace=False)])

# predição: média dos modelos (1 com final.pt)
p_cnn = np.zeros(len(idx))
for ck in cks:
    m = tc.criar_modelo(ck['args']['backbone'], len(ck['canais']), ck['args']['patch'], pretrained=False)
    m.load_state_dict(ck['estado'])
    m.to(dispositivo)
    ds = tc.Patches(X_v, meta_v, idx, list(ck['canais']), ck['est'], ck['args']['patch'], aumentar=False)
    p_cnn += tc.prever(m, DataLoader(ds, 32), dispositivo) / len(cks)

tab = meta_v.loc[idx, ['id', 'celula', 'origem_amostra', 'codigo_obj', 'p_rf', 'p_gtb']].copy()
tab['referencia'] = np.where(tab.origem_amostra == 'ocorrencia_teste', 'fosfato', 'fundo')
tab['p_cnn'] = p_cnn
print(tab[['id', 'celula', 'referencia', 'codigo_obj', 'p_rf', 'p_gtb', 'p_cnn']].round(3).to_string(index=False))

# figura: um patch por quadro, canal escolhido (padrão: o primeiro do modelo)
c = canais_v.index(CANAL_V) if CANAL_V in canais_v else 0
ncol = 5
nlin = int(np.ceil(len(idx) / ncol))
fig, axs = plt.subplots(nlin, ncol, figsize=(3.2 * ncol, 3.6 * nlin), squeeze=False)
P, pp = X_v.shape[-1], args_m['patch']
for ax, i, r, pc in zip(axs.flat, idx, tab.itertuples(), p_cnn):
    img = X_v[i, c].astype(np.float32)
    lo, hi = np.nanpercentile(img, [2, 98]) if np.isfinite(img).any() else (0, 1)
    ax.imshow(img, cmap='magma', vmin=lo, vmax=hi)
    ax.plot(P // 2, P // 2, 'x', color='cyan', ms=10, mew=2)
    if pp < P:
        o = (P - pp) // 2
        ax.add_patch(Rectangle((o - .5, o - .5), pp, pp, fill=False, ls='--', ec='cyan', lw=1))
    cor = 'darkgreen' if (pc >= 0.5) == (r.referencia == 'fosfato') else 'firebrick'
    ax.set_title(f'{r.referencia.upper()} · {r.celula}\n'
                 f'RF {r.p_rf:.2f} · GTB {r.p_gtb:.2f}\nCNN {pc:.2f}', fontsize=9, color=cor)
    ax.axis('off')
for ax in axs.flat[len(idx):]:
    ax.axis('off')
fig.suptitle(f'{arquivos[0].parent.name} · canal {canais_v[c]} · título verde = CNN acerta com corte 0,5',
             fontsize=11)
plt.tight_layout()

# todas as ocorrências e fundo do teste (não só as do quadro): onde a CNN põe cada grupo
if (pasta_t / 'teste.csv').exists():
    t = pd.read_csv(pasta_t / 'teste.csv')
    ref = np.where(t.origem_amostra == 'ocorrencia_teste', 'fosfato', np.where(t.origem_amostra == 'fundo', 'fundo', None))
    print('\np CNN no teste inteiro (teste.csv):')
    print(t.assign(referencia=ref).dropna(subset=['referencia']).groupby('referencia').p
           .describe(percentiles=[.1, .5, .9]).round(3).to_string())

`[10b]`

## Todas as presenças de fosfato × predição das rodadas

Passa **todos** os pontos de fosfato (ocorrências SGB, só o ponto sem jitter) e o
fundo aleatório por cada rodada treinada e grava uma linha por ponto e grupo em
`runs_cnn/comparacao_presencas.csv`:

| grupo | rodada de validação cruzada (`fold*.pt`) | rodada `_teste` (`final.pt`) |
|---|---|---|
| **treino** | média dos folds que treinaram com o ponto | o modelo final (viu o ponto) |
| **validação** | só o modelo do fold que deixou o ponto de fora | — |
| **teste** | média dos 5 folds (células nunca vistas) | o modelo final (não viu) |

`treino` mostra quanto a rede decorou; `validação` e `teste` são as leituras
honestas. O fundo entra para comparar: um p alto no fosfato só vale se o fundo
ficar baixo. `uso_treino = 0` marca ocorrências fora da magenta, que não entraram
no treino mesmo no grupo `treino`. RF e GTB (`p_rf`, `p_gtb`) vão nas mesmas
linhas — lembre que eles foram treinados com todos os pontos.

Retomável: rodada já na tabela é pulada (`REFAZER` refaz tudo). A célula 12 só lê
a tabela e desenha.

In [ ]:
# [11] p das rodadas em todas as presenças de fosfato (treino, validação, teste) e no fundo
#@markdown Rodadas separadas por vírgula; vazio = todas as pastas de RUNS_DIR com .pt
RODADAS_C = ''  #@param {type:"string"}
REFAZER = False  #@param {type:"boolean"}

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader
import treinar_cnn as tc

TABELA = Path(RUNS_DIR) / 'comparacao_presencas.csv'
antiga = pd.read_csv(TABELA) if TABELA.exists() and not REFAZER else pd.DataFrame()
feitas = set(antiga.rodada) if len(antiga) else set()
pedidas = [r.strip() for r in RODADAS_C.split(',') if r.strip()]
pastas = [p for p in sorted(Path(RUNS_DIR).iterdir())
          if p.is_dir() and any(p.glob('*.pt')) and (not pedidas or p.name in pedidas)]
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COLUNAS = ['id', 'celula', 'conjunto', 'fold', 'origem_amostra', 'jitter', 'uso_treino',
           'codigo_obj', 'lon', 'lat', 'p_rf', 'p_gtb']
base_dados, novas = {}, []

for pasta in pastas:
    if pasta.name in feitas:
        print(f'{pasta.name}: já na tabela — pulada')
        continue
    arquivos = [pasta / 'final.pt'] if (pasta / 'final.pt').exists() else sorted(pasta.glob('fold*.pt'))
    cks = {a.stem: torch.load(a, map_location='cpu', weights_only=False) for a in arquivos}
    a0 = next(iter(cks.values()))['args']
    if a0['modelo'] not in base_dados:          # X de cada modelo (A/B1) carregado uma vez só
        base_dados[a0['modelo']] = tc.carregar(a0['modelo'], Path(globals().get('dados', PATCHES_DIR)))
    X, meta, canais = base_dados[a0['modelo']]
    ocorr = meta.origem_amostra.str.startswith('ocorrencia')
    idx = meta.index[(ocorr & (meta.jitter == 0)) | (meta.origem_amostra == 'fundo')].to_numpy()

    P = {}                                      # arquivo → p em cada patch de idx
    for nome, ck in cks.items():
        m = tc.criar_modelo(ck['args']['backbone'], len(ck['canais']), ck['args']['patch'], pretrained=False)
        m.load_state_dict(ck['estado'])
        m.to(dispositivo)
        ds = tc.Patches(X, meta, idx, list(ck['canais']), ck['est'], ck['args']['patch'], aumentar=False)
        P[nome] = tc.prever(m, DataLoader(ds, 512), dispositivo)
        del m
    if dispositivo.type == 'cuda':
        torch.cuda.empty_cache()

    t = meta.loc[idx, [c for c in COLUNAS if c in meta]].reset_index(drop=True)
    t['referencia'] = np.where(t.origem_amostra == 'fundo', 'fundo', 'fosfato')
    teste = (t.conjunto == 'teste').to_numpy()
    if 'final' in P:
        partes = [t.assign(grupo=np.where(teste, 'teste', 'treino'), p_cnn=P['final'])]
    else:
        folds = {int(n[4:]): v for n, v in P.items()}          # fold<k> → p
        f = t.fold.to_numpy()
        fora = np.array([folds[k][i] if k in folds else np.nan for i, k in enumerate(f)])
        dentro = np.array([np.mean([v[i] for k2, v in folds.items() if k2 != k]) if len(folds) > 1 else np.nan
                           for i, k in enumerate(f)])
        partes = [t[teste].assign(grupo='teste', p_cnn=np.mean(list(folds.values()), axis=0)[teste]),
                  t[~teste].assign(grupo='validação', p_cnn=fora[~teste]),
                  t[~teste].assign(grupo='treino', p_cnn=dentro[~teste])]
    t = pd.concat(partes, ignore_index=True).dropna(subset=['p_cnn'])
    t.insert(0, 'rodada', pasta.name)
    t.insert(1, 'modelo', a0['modelo'])
    t.insert(2, 'backbone', a0['backbone'])
    novas.append(t)
    print(f'{pasta.name}: {len(cks)} modelo(s) · {int((t.referencia == "fosfato").sum())} linhas de fosfato, '
          f'{int((t.referencia == "fundo").sum())} de fundo')

tabela = pd.concat([antiga, *novas], ignore_index=True) if novas else antiga
assert len(tabela), 'nenhuma rodada com .pt em RUNS_DIR'
tabela.to_csv(TABELA, index=False)

# resumo: AUC fosfato × fundo por rodada e grupo (RF e GTB nas mesmas linhas)
def auc(g, col):
    y = (g.referencia == 'fosfato').astype(int)
    return roc_auc_score(y, g[col]) if y.nunique() == 2 else np.nan

linhas = []
for (rodada, grupo), g in tabela.groupby(['rodada', 'grupo']):
    fo = g[g.referencia == 'fosfato']
    linhas.append({'rodada': rodada, 'grupo': grupo, 'n_fosfato': len(fo),
                   'p_fosfato': fo.p_cnn.median(), 'p_fundo': g[g.referencia == 'fundo'].p_cnn.median(),
                   'auc_cnn': auc(g, 'p_cnn'), 'auc_rf': auc(g, 'p_rf'), 'auc_gtb': auc(g, 'p_gtb')})
RESUMO = pd.DataFrame(linhas)
RESUMO.to_csv(Path(RUNS_DIR) / 'comparacao_presencas_auc.csv', index=False)
print(f'\n{TABELA}  ({len(tabela)} linhas)')
print(RESUMO.round(3).to_string(index=False))

In [ ]:
# [12] gráficos da comparação (lê comparacao_presencas.csv)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

tabela = pd.read_csv(Path(RUNS_DIR) / 'comparacao_presencas.csv')
RESUMO = pd.read_csv(Path(RUNS_DIR) / 'comparacao_presencas_auc.csv')
AZUL, LARANJA, AQUA, CINZA, TEXTO = '#2a78d6', '#eb6834', '#1baf7a', '#9a9993', '#52514e'
GRUPOS = [g for g in ('treino', 'validação', 'teste') if g in set(tabela.grupo)]
rodadas = sorted(tabela.rodada.unique())

# 1) AUC por rodada: validação e teste (as leituras honestas); RF/GTB do teste como referência
fig, ax = plt.subplots(figsize=(max(7, 0.9 * len(rodadas)), 4.2))
x = np.arange(len(rodadas))
for grupo, cor, dx in [('validação', AZUL, -0.12), ('teste', LARANJA, 0.12)]:
    r = RESUMO[RESUMO.grupo == grupo].set_index('rodada').reindex(rodadas)
    ax.plot(x + dx, r.auc_cnn, 'o', ms=8, color=cor, mec='white', mew=1.5, label=f'CNN · {grupo}')
te = RESUMO[RESUMO.grupo == 'teste']
for col, nome, ls in [('auc_gtb', 'GTB', '--'), ('auc_rf', 'RF', ':')]:
    if te[col].notna().any():
        v = te[col].median()
        ax.axhline(v, color=CINZA, ls=ls, lw=1.5, zorder=0)
        ax.text(len(rodadas) - 0.45, v, f'{nome} teste {v:.2f} ', va='bottom', ha='right', color=TEXTO, fontsize=8)
ax.axhline(0.5, color=CINZA, lw=0.8, zorder=0)
ax.set_xticks(x, rodadas, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('AUC fosfato × fundo')
ax.set_ylim(0.4, 1)
ax.set_xlim(-0.6, len(rodadas) - 0.4)
ax.grid(axis='y', color='#e6e5e0', lw=0.8)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc='lower left')
ax.set_title('Separação entre presenças de fosfato e fundo por rodada', fontsize=11, loc='left')
plt.tight_layout()
plt.show()

# 2) distribuição do p: fosfato × fundo, um painel por grupo; RF e GTB no fim como referência
fig, axs = plt.subplots(len(GRUPOS), 1, figsize=(max(8, 1.0 * (len(rodadas) + 2)), 3.2 * len(GRUPOS)),
                        sharey=True, squeeze=False)
nomes = rodadas + ['RF', 'GTB']
for ax, grupo in zip(axs[:, 0], GRUPOS):
    g = tabela[tabela.grupo == grupo]
    ref = g.drop_duplicates('id')                # RF/GTB: um valor por ponto
    for ref_nome, cor, dx in [('fosfato', AZUL, -0.18), ('fundo', CINZA, 0.18)]:
        dados = [g[(g.rodada == r) & (g.referencia == ref_nome)].p_cnn.to_numpy() for r in rodadas]
        dados += [ref[ref.referencia == ref_nome][c].dropna().to_numpy() for c in ('p_rf', 'p_gtb')]
        pos = [i + dx for i, d in enumerate(dados) if len(d)]
        bp = ax.boxplot([d for d in dados if len(d)], positions=pos, widths=0.3, patch_artist=True,
                        showfliers=False, medianprops={'color': 'white', 'lw': 1.5})
        for caixa in bp['boxes']:
            caixa.set(facecolor=cor, edgecolor=cor)
        for parte in ('whiskers', 'caps'):
            for l in bp[parte]:
                l.set(color=cor, lw=1)
    ax.axvline(len(rodadas) - 0.5, color=CINZA, lw=0.8)
    ax.set_xticks(range(len(nomes)), nomes, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('p')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', color='#e6e5e0', lw=0.8)
    ax.spines[['top', 'right']].set_visible(False)
    n = int(g[g.referencia == 'fosfato'].id.nunique())
    ax.set_title(f'{grupo} · {n} pontos de fosfato', fontsize=10, loc='left', color=TEXTO)
axs[0, 0].legend([plt.Rectangle((0, 0), 1, 1, color=AZUL), plt.Rectangle((0, 0), 1, 1, color=CINZA)],
                 ['fosfato (ocorrência SGB)', 'fundo aleatório'], frameon=False, fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

`[9b]`

## Próximo passo: mapa

As rodadas ficam em `RUNS_DIR`. A predição (janela deslizante sobre os tiles
exportados por `exportar_tiles_drive.py`) roda em outro notebook.